# Get reduced word with the best longest strictly increasing suffix (SIS)

In [1]:
# note that Sage's SignedPermutations() and .reduced_words() use different conventions than we do, which
# changes the length and reduced words of the permutations

# Example below
n = 3
Bn = SignedPermutations(n)
w = Bn([-3,-1,2])
print("using Sage's SignedPermutations(n):")
print(f"w = {w}")
print(f"l(w) = {w.length()}")
preReducedWords = w.reduced_words()
print(f"reduced words where you work right-to-left and s_3 turns w(3) negative: {preReducedWords}")
print("\n we would say l(w) = 4 and the reduced words were [0, 2, 1, 0] and [2, 0, 1, 0], working left-to-right with s_0 turning w(1) negative")

using Sage's SignedPermutations(n):
w = [-3, -1, 2]
l(w) = 6
reduced words where you work right-to-left and s_3 turns w(3) negative: [[2, 1, 3, 2, 1, 3], [2, 3, 1, 2, 1, 3], [2, 1, 3, 2, 3, 1], [2, 3, 2, 1, 2, 3], [2, 3, 1, 2, 3, 1]]

 we would say l(w) = 4 and the reduced words were [0, 2, 1, 0] and [2, 0, 1, 0], working left-to-right with s_0 turning w(1) negative


In [2]:
# Modified from Gemini 3.1 Pro: get reduced words the way we think of them
def rws(w):
    """
    Returns all reduced words for a signed permutation 'w' using the convention:
      - s_0 negates the first element (index 0).
      - s_i (for i > 0) swaps elements at indices i-1 and i.
    """
    w_tuple = tuple(w)
    n = len(w_tuple)
    identity = tuple(range(1, n + 1))
    
    # 1. Define the length function for this specific convention
    def length(v):
        inv = sum(1 for i in range(n) for j in range(i + 1, n) if v[i] > v[j])
        neg = sum(1 for i in range(n) for j in range(i, n) if v[i] + v[j] < 0)
        return inv + neg

    # 2. Define the generator actions on the right (swapping positions)
    def apply_generator(v, i):
        res = list(v)
        if i == 0:
            res[0] = -res[0]
        else:
            res[i-1], res[i] = res[i], res[i-1]
        return tuple(res)

    # 3. Backtracking with memoization to find all paths of decreasing length
    memo = {identity: [[]]}

    def search(v):
        if v in memo:
            return memo[v]
        
        curr_len = length(v)
        words = []
        for i in range(n):
            prev_v = apply_generator(v, i)
            if length(prev_v) == curr_len - 1:
                # Moving to prev_v is a valid step down the Coxeter weak order
                for word in search(prev_v):
                    words.append(word + [i])
        
        memo[v] = words
        return words

    return search(w_tuple)

In [3]:
# Test 1 (with an example Judy did in her notes)
w = [-3, 1, -2, 5, 4]
words = rws(w)

print(f"Total reduced words: {len(words)}")
print(f"Is your word in the list? {[1, 0, 2, 1, 0, 2, 4] in words}")
print(f"\nFirst {min(len(words),5)} reduced words:")
for word in words[:5]:
    print(word)

Total reduced words: 35
Is your word in the list? True

First 5 reduced words:
[4, 1, 0, 1, 2, 1, 0]
[1, 4, 0, 1, 2, 1, 0]
[1, 0, 4, 1, 2, 1, 0]
[1, 0, 1, 4, 2, 1, 0]
[1, 0, 1, 2, 4, 1, 0]


In [4]:
# Test 2 (with an example I've done by hand)
w = [-3, -1, 2]
words = rws(w)

print(f"Total reduced words: {len(words)}")
print(f"Is your word in the list? {[0, 2, 1, 0] in words}")
print(f"\nFirst {min(len(words),5)} reduced words:")
for word in words[:5]:
    print(word)

Total reduced words: 2
Is your word in the list? True

First 2 reduced words:
[2, 0, 1, 0]
[0, 2, 1, 0]


In [5]:
# modified from Gemini 3.1 Pro: from a list of words, pick one with the longest strictly increasing suffix (SIS)
import itertools
def choose_longest_SIS(list_of_lists):
    """
    Finds the list with the longest strictly increasing suffix.
    In the event of a tie, returns the lexicographically smallest list.
    Assumes all sublists are of equal length.
    """
    def get_suffix_length(lst):
        if not lst:
            return 0
        length = 1
        for i in range(len(lst) - 1, 0, -1):
            if lst[i-1] < lst[i]:
                length += 1
            else:
                break
        return length

    if not list_of_lists:
        return None
        
    # min() evaluates the tuple:
    # 1. -get_suffix_length(lst): Minimizing a negative maximizes the actual length.
    # 2. lst: Minimizes the list itself (lexicographically smallest).
    bestList = min(list_of_lists, key=lambda lst: (-get_suffix_length(lst), lst))
    return (bestList, get_suffix_length(bestList))

In [6]:
# Returns the reduced word with the longest SIS and its SIS length
def canonical_word(w):
    redWords = rws(w)
    return choose_longest_SIS(redWords)

In [7]:
# test
w = [-3, -1, 2]
canonical_word(w)

([0, 2, 1, 0], 1)

# maximal bottom row

In [8]:
# given a vector t, find the maximal compatible bottom sequence b
def maxl_b(t,vec=False):
    l = len(t)-1
    # last entry of b is equal to last entry of t
    maxlB = list(range(0,l)) + [int(t[l])]
    predet = False
    # for the rest of the entries...
    for i in range(l-1,-1,-1):
        if predet == False:
            # figure out if t_i or b_{i+1} is bounding b_i from above
            maxPossible = min(int(maxlB[i+1]),int(t[i]))
            # if that upper bound is bigger than 1, check if we have a forced increase
            if maxPossible > 0 and t[i]<t[i+1]:
                maxlB[i] = min(int(maxlB[i+1])-1,int(t[i]))
            # if that upper bound is less than 1, check if we have the peak condition
            elif i>0 and maxPossible <= 0 and t[i-1]<t[i]>=t[i+1]:
                maxlB[i] = min(int(maxlB[i+1]),int(t[i]))
                # if we do have a peak and b_i and b_{i+1} are the same, then we need to make b_{i-1} at most (b_i)-1
                if maxlB[i] == maxlB[i+1]:
                    predet = True
            else:
                maxlB[i] = maxPossible
        # deal with a peak if we have
        else:
            maxlB[i] = min(int(maxlB[i+1]-1),int(t[i]))
            predet = False
    if vec == True:
        print(matrix([list(t),maxlB]))
    return maxlB

In [9]:
# test
print("finding a maximal compatible b for the given vector:")
v = [0, 1, 2, 0, 1, 2]
maxl_b(v,True)

finding a maximal compatible b for the given vector:
[ 0  1  2  0  1  2]
[-1 -1  0  0  1  2]


[-1, -1, 0, 0, 1, 2]

# Lehmer code stuff

In [10]:
# Lehmer code and signed Lehmer code
def lehmer_code(w):
    """
    Return the Lehmer code of the word w.

    L_i = #{j > i : w[i] > w[j]}
    """
    n = len(w)
    return vector([
        sum(1 for j in range(i+1, n) if w[i] > w[j])
        for i in range(n)
    ])

def N(w):
    """
    Return N(w) = (N_1 > N_2 > ...), where
    N_i are the positive integers whose inverse image under the
    signed permutation w is negative.
    """
    return vector(sorted([-x for x in w if x < 0], reverse=False)) # I changed the reverse to False to make signed_Lehmer match the paper

def signed_Lehmer(w):
    return (N(w),lehmer_code(w))

# modified from Gemini 3.5 Thinking: get the number of nonzero entries in the right half of the Lehmer code plus 1 if the left half has nonzero entries
def Lcw_metric(tuple_of_tuples):
    """
    Calculates n based on a tuple of tuples:
    - If the first tuple has a nonzero entry: 
        returns (number of nonzero entries in the second tuple) + 1
    - Otherwise:
        returns 0
    """
    # Unpack the first two tuples
    t1, t2 = tuple_of_tuples[0], tuple_of_tuples[1]
    
    # Count the nonzero entries in the second tuple
    nonzero_count_t2 = sum(1 for x in t2 if x != 0)
    
    # Check if the first tuple contains any nonzero element
    if any(x != 0 for x in t1):
        return nonzero_count_t2 + 1
    else:
        return nonzero_count_t2

In [11]:
# test 1
"""
For $u = \\overline{1}35\\overline{4}2$, then $L(u) = (1,2,2,0,0)$ and $N(u) = (4,1)$, so the signed Lehmer code $Lc(u) = (1,4\\bigg|1,1,2,0,0)$.
"""
w = [-1,3,5,-4,2]
print(f"w = {w}")
print(f"Lc(w) = {lehmer_code(w)}")
print(f"N(w) = {N(w)}")
Lcw = signed_Lehmer(w)
print(f"signed Lehmer code of w: {Lcw}")
print(f"Lehmer code metric: {Lcw_metric(Lcw)}")

# test 2
w = [2,1]
print(f"\nw = {w}")
print(f"Lc(w) = {lehmer_code(w)}")
print(f"N(w) = {N(w)}")
Lcw = signed_Lehmer(w)
print(f"signed Lehmer code of w: {Lcw}")
print(f"Lehmer code metric: {Lcw_metric(Lcw)}")

w = [-1, 3, 5, -4, 2]
Lc(w) = (1, 2, 2, 0, 0)
N(w) = (1, 4)
signed Lehmer code of w: ((1, 4), (1, 2, 2, 0, 0))
Lehmer code metric: 4

w = [2, 1]
Lc(w) = (1, 0)
N(w) = ()
signed Lehmer code of w: ((), (1, 0))
Lehmer code metric: 1


# the actual data we want to anazlyze

In [12]:
# functions to print the data we want in a nice legible format
def print_data(w):
    print(f"w = {w}")
    Lcw = signed_Lehmer(w)
    print(f"\nLc(w) = {Lcw}")
    Lcw_sum = Lcw_metric(Lcw)
    print(f"Lcw_sum = {Lcw_sum}")
    (rw,SISlength) = canonical_word(w)
    print(f"\nreduced word = {rw}")
    if rw != []:
        print(f"b = {maxl_b(rw)}")
        print(f"Longest SIS length = {SISlength}")
        print(f"\nDoes Lcs_sum = Longest SIS length? {Lcw_sum == SISlength}")

# Modified from Gemini 3.1 Pro: helper function to print Lehmer code nicely
def tuple_format(t):
    # Unpack the two tuples
    t1, t2 = t
    
    # Convert elements to strings and join them with commas
    str1 = ", ".join(map(str, t1))
    str2 = ", ".join(map(str, t2))
    
    # Print the formatted string
    return f"({str1} | {str2})"

# --- Example Usage ---
# my_tuples = ((1, 2, 'a'), (3.14, 'x', 'y'))
# print_tuple_format(my_tuples)
# Output: $(1, 2, a | 3.14, x, y)$

def print_data_LaTeX(w):
    print(f"$w = {w}$")
    Lcw = signed_Lehmer(w)
    print(f"\n$\\Lc(w) = {tuple_format(Lcw)}$\\\\")
    Lcw_sum = Lcw_metric(Lcw)
    print(f"$\\LcSum(w) = {Lcw_sum}$")
    (rw,SISlength) = canonical_word(w)
    print(f"\nreduced word: ${rw}$\\\\")
    if rw != []:
        print(f"$b = {maxl_b(rw)}$\\\\")
        print(f"Longest SIS length: ${SISlength}$\\\\")
        print(f"\nDoes $\\LcSum(w)$ = Longest SIS length? {Lcw_sum == SISlength}")

In [13]:
# test
print_data([-3,-2,1])

print("\n")

print_data_LaTeX([-3,-2,1])

w = [-3, -2, 1]

Lc(w) = ((2, 3), (0, 0, 0))
Lcw_sum = 1

reduced word = [1, 0, 2, 1, 0]
b = [-1, -1, 0, 0, 0]
Longest SIS length = 1

Does Lcs_sum = Longest SIS length? True


$w = [-3, -2, 1]$

$\Lc(w) = (2, 3 | 0, 0, 0)$\\
$\LcSum(w) = 1$

reduced word: $[1, 0, 2, 1, 0]$\\
$b = [-1, -1, 0, 0, 0]$\\
Longest SIS length: $1$\\

Does $\LcSum(w)$ = Longest SIS length? True


In [19]:
def test(n):
    currentFlat = [i in range(1,n+1)]
    print(f"\\subsubsection{{{currentFlat}}}")
    print(f"\nStarting permutations with $\\flatt(w) = {currentFlat}$")
    for w in SignedPermutations(n):
        absW = [abs(w[i]) for i in range(len(w))]
        if absW != currentFlat:
            currentFlat = absW
            print("\n")
            print("\\hrule")
            print("\n")
            print(f"\\subsubsection{{{currentFlat}}}")
            print(f"\nStarting permutations with $\\flat(w) = {currentFlat}$")
        print("\n")
        print("\\hrule")
        print("\n")
        print_data_LaTeX(w)
    print("\n")
    print("\\hrule")
    print("\n")

# Data collection

In [20]:
%%time
test(1)

\subsubsection{[False]}

Starting permutations with $\flatt(w) = [False]$


\hrule


\subsubsection{[1]}

Starting permutations with $\flat(w) = [1]$


\hrule


$w = [1]$

$\Lc(w) = ( | 0)$\\
$\LcSum(w) = 0$

reduced word: $[]$\\


\hrule


$w = [-1]$

$\Lc(w) = (1 | 0)$\\
$\LcSum(w) = 1$

reduced word: $[0]$\\
$b = [0]$\\
Longest SIS length: $1$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


CPU times: user 4.71 ms, sys: 0 ns, total: 4.71 ms
Wall time: 4.72 ms


In [21]:
%%time
test(2)

\subsubsection{[False]}

Starting permutations with $\flatt(w) = [False]$


\hrule


\subsubsection{[1, 2]}

Starting permutations with $\flat(w) = [1, 2]$


\hrule


$w = [1, 2]$

$\Lc(w) = ( | 0, 0)$\\
$\LcSum(w) = 0$

reduced word: $[]$\\


\hrule


$w = [1, -2]$

$\Lc(w) = (2 | 1, 0)$\\
$\LcSum(w) = 2$

reduced word: $[1, 0, 1]$\\
$b = [0, 0, 1]$\\
Longest SIS length: $2$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


$w = [-1, 2]$

$\Lc(w) = (1 | 0, 0)$\\
$\LcSum(w) = 1$

reduced word: $[0]$\\
$b = [0]$\\
Longest SIS length: $1$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


$w = [-1, -2]$

$\Lc(w) = (1, 2 | 1, 0)$\\
$\LcSum(w) = 2$

reduced word: $[0, 1, 0, 1]$\\
$b = [-1, 0, 0, 1]$\\
Longest SIS length: $2$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


\subsubsection{[2, 1]}

Starting permutations with $\flat(w) = [2, 1]$


\hrule


$w = [2, 1]$

$\Lc(w) = ( | 1, 0)$\\
$\LcSum(w) = 1$

reduced word: $[1]$\\
$b = [1]$\\
Longest SIS length: $1$\\


In [22]:
%%time
test(3)

\subsubsection{[False]}

Starting permutations with $\flatt(w) = [False]$


\hrule


\subsubsection{[1, 2, 3]}

Starting permutations with $\flat(w) = [1, 2, 3]$


\hrule


$w = [1, 2, 3]$

$\Lc(w) = ( | 0, 0, 0)$\\
$\LcSum(w) = 0$

reduced word: $[]$\\


\hrule


$w = [1, 2, -3]$

$\Lc(w) = (3 | 1, 1, 0)$\\
$\LcSum(w) = 3$

reduced word: $[2, 1, 0, 1, 2]$\\
$b = [0, 0, 0, 1, 2]$\\
Longest SIS length: $3$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


$w = [1, -2, 3]$

$\Lc(w) = (2 | 1, 0, 0)$\\
$\LcSum(w) = 2$

reduced word: $[1, 0, 1]$\\
$b = [0, 0, 1]$\\
Longest SIS length: $2$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


$w = [1, -2, -3]$

$\Lc(w) = (2, 3 | 2, 1, 0)$\\
$\LcSum(w) = 3$

reduced word: $[1, 0, 1, 2, 1, 0, 1, 2]$\\
$b = [-1, -1, -1, 0, 0, 0, 1, 2]$\\
Longest SIS length: $3$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


$w = [-1, 2, 3]$

$\Lc(w) = (1 | 0, 0, 0)$\\
$\LcSum(w) = 1$

reduced word: $[0]$\\
$b = [0]$\\
Longest SIS length: 

In [23]:
%%time
test(4)

\subsubsection{[False]}

Starting permutations with $\flatt(w) = [False]$


\hrule


\subsubsection{[1, 2, 3, 4]}

Starting permutations with $\flat(w) = [1, 2, 3, 4]$


\hrule


$w = [1, 2, 3, 4]$

$\Lc(w) = ( | 0, 0, 0, 0)$\\
$\LcSum(w) = 0$

reduced word: $[]$\\


\hrule


$w = [1, 2, 3, -4]$

$\Lc(w) = (4 | 1, 1, 1, 0)$\\
$\LcSum(w) = 4$

reduced word: $[3, 2, 1, 0, 1, 2, 3]$\\
$b = [0, 0, 0, 0, 1, 2, 3]$\\
Longest SIS length: $4$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


$w = [1, 2, -3, 4]$

$\Lc(w) = (3 | 1, 1, 0, 0)$\\
$\LcSum(w) = 3$

reduced word: $[2, 1, 0, 1, 2]$\\
$b = [0, 0, 0, 1, 2]$\\
Longest SIS length: $3$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


$w = [1, 2, -3, -4]$

$\Lc(w) = (3, 4 | 2, 2, 1, 0)$\\
$\LcSum(w) = 4$

reduced word: $[2, 1, 0, 1, 2, 3, 2, 1, 0, 1, 2, 3]$\\
$b = [-1, -1, -1, -1, -1, 0, 0, 0, 0, 1, 2, 3]$\\
Longest SIS length: $4$\\

Does $\LcSum(w)$ = Longest SIS length? True


\hrule


$w = [1, -2, 3, 4]$

$\Lc(w) = (2 | 